<a href="https://colab.research.google.com/github/endahesthiningtyas/Bioinfor/blob/main/Endah_Esthiningtyas_Kinetic_Modelling_UAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UAS Bioteknologi - Topik Biologi Sistem
# Nama : Endah Esthiningtyas
# NIM : 23/519190/BI/11310
# Kelas : A

## Pemodelan Kinetik, Ujian Akhir Semester Genap T.A. 2025/2026

---

Skrip ini mendemonstrasikan cara memodelkan dan mensimulasikan jaringan metabolik 4-reaksi menggunakan **Persamaan Diferensial Biasa (ODE)**. Sistem ini mencakup kinetika **Michaelis-Menten** pada langkah pertama (v₁) serta **inhibisi umpan balik alosterik non-kompetitif** oleh produk akhir (P) terhadap v₁.

Topologi jaringan yang dimodelkan adalah sebagai berikut:

```
X --[v1]--> A --[v2]--> B --[v3]--> P
                |                    |
               [v4]         (inhibisi alosterik)
                |                    |
            Byproduct <--------------+
```

- **v₁**: X → A (Michaelis-Menten, dihambat secara non-kompetitif oleh P)
- **v₂**: A → B (kinetika orde pertama)
- **v₃**: B → P (kinetika orde pertama)
- **v₄**: A → Byproduct (kinetika orde pertama, jalur pelarian)


## Tujuan Pembelajaran

Dengan menjalankan dan menganalisis skrip ini, mahasiswa diharapkan dapat:

1. Memahami cara memodelkan jaringan metabolik menggunakan sistem ODE.
2. Membangun **matriks stoikiometri (S)** dari topologi jaringan reaksi.
3. Merumuskan persamaan diferensial kinetik untuk setiap metabolit internal.
4. Mengimplementasikan model **inhibisi non-kompetitif** dalam persamaan Michaelis-Menten.
5. Menyelesaikan sistem ODE secara numerik menggunakan Python.
6. Memvisualisasikan dinamika konsentrasi metabolit sepanjang siklus fermentasi 48 jam.

## Instalasi Library

Pastikan library yang dibutuhkan sudah tersedia di lingkungan Colab.

In [1]:
# Install altair jika belum tersedia
!pip install altair --quiet

## Import Library

In [2]:
import numpy as np
import altair as alt
import pandas as pd
from scipy.integrate import odeint

## Matriks Stoikiometri (S)

Sebelum merumuskan ODE, kita definisikan **matriks stoikiometri S** yang merepresentasikan bagaimana setiap reaksi mempengaruhi setiap metabolit internal.

Konvensi: **+1** = diproduksi, **−1** = dikonsumsi, **0** = tidak terlibat

|            | v₁ | v₂ | v₃ | v₄ |
|------------|----|----|----|----|  
| Metabolit A| +1 | −1 |  0 | −1 |
| Metabolit B|  0 | +1 | −1 |  0 |
| Produk P   |  0 |  0 | +1 |  0 |

Dari matriks ini, kita dapat merumuskan persamaan diferensial:

$$\frac{d[A]}{dt} = v_1 - v_2 - v_4$$

$$\frac{d[B]}{dt} = v_2 - v_3$$

$$\frac{d[P]}{dt} = v_3$$

In [3]:
# Visualisasi matriks stoikiometri
S = np.array([
    [+1, -1,  0, -1],   # Metabolit A
    [ 0, +1, -1,  0],   # Metabolit B
    [ 0,  0, +1,  0],   # Produk P
])

metabolites = ['Metabolit A', 'Metabolit B', 'Produk P']
reactions   = ['v1', 'v2', 'v3', 'v4']

df_S = pd.DataFrame(S, index=metabolites, columns=reactions)
print("Matriks Stoikiometri S:")
print(df_S)

Matriks Stoikiometri S:
             v1  v2  v3  v4
Metabolit A   1  -1   0  -1
Metabolit B   0   1  -1   0
Produk P      0   0   1   0


## Model Kinetik

### Laju Reaksi

Setiap laju reaksi dimodelkan sebagai berikut:

**v₁,  Michaelis-Menten dengan inhibisi non-kompetitif oleh P:**

$$v_1 = \frac{V_{1,max} \cdot [X]}{(K_{m1} + [X]) \cdot \left(1 + \frac{[P]}{K_i}\right)}$$

Inhibitor P berikatan pada situs alosterik dan **mengurangi V₁,max efektif** tanpa mengubah afinitas substrat (Kₘ). Ketika [P] = 0, ekspresi ini menjadi kinetika Michaelis-Menten standar.

**v₂, v₃, v₄,  Kinetika orde pertama:**

$$v_2 = k_2 \cdot [A], \quad v_3 = k_3 \cdot [B], \quad v_4 = k_4 \cdot [A]$$

### Parameter Sistem

| Parameter | Keterangan | Nilai |
|-----------|-----------|-------|
| V₁,max | Laju maksimum v₁ | 5.0 |
| Kₘ₁ | Konstanta Michaelis untuk v₁ | 2.0 |
| Kᵢ | Konstanta inhibisi | 3.0 |
| X | Konsentrasi substrat eksternal (konstan) | 10 |
| k₂ | Konstanta laju orde-1 untuk A → B | 1.0 |
| k₃ | Konstanta laju orde-1 untuk B → P | 0.8 |
| k₄ | Konstanta laju orde-1 untuk A → Byproduct | 0.3 |

## Definisi Sistem ODE

Fungsi berikut mendefinisikan sistem ODE yang akan diselesaikan secara numerik.

Persamaan ODE lengkap:

$$\frac{d[A]}{dt} = \frac{V_{1,max} \cdot [X]}{(K_{m1}+[X])\left(1+\frac{[P]}{K_i}\right)} - k_2[A] - k_4[A]$$

$$\frac{d[B]}{dt} = k_2[A] - k_3[B]$$

$$\frac{d[P]}{dt} = k_3[B]$$

In [4]:
def network(y, t):
    # Buat container untuk menyimpan turunan
    dydt = np.empty(3)

    # --- Parameter sistem ---
    V1max = 5.0   # Laju maksimum v1
    Km1   = 2.0   # Konstanta Michaelis untuk v1
    Ki    = 3.0   # Konstanta inhibisi alosterik
    X     = 10    # Konsentrasi substrat eksternal (konstan)
    k2    = 1.0   # Konstanta laju A → B
    k3    = 0.8   # Konstanta laju B → P
    k4    = 0.3   # Konstanta laju A → Byproduct

    # --- Variabel dinamik ---
    A = y[0]   # Konsentrasi metabolit A
    B = y[1]   # Konsentrasi metabolit B
    P = y[2]   # Konsentrasi produk P

    # --- Laju reaksi ---
    # v1: Michaelis-Menten dengan inhibisi non-kompetitif oleh P
    v1 = (V1max * X) / ((Km1 + X) * (1 + P / Ki))

    # v2, v3, v4: kinetika orde pertama
    v2 = k2 * A
    v3 = k3 * B
    v4 = k4 * A

    # --- Persamaan diferensial (dari matriks stoikiometri S) ---
    dydt[0] = v1 - v2 - v4   # d[A]/dt
    dydt[1] = v2 - v3         # d[B]/dt
    dydt[2] = v3              # d[P]/dt

    return dydt

## Kondisi Awal dan Penyelesaian ODE

Kondisi awal ditetapkan dengan semua metabolit internal bernilai nol (sel belum memproduksi apa-apa). Simulasi dijalankan selama **48 jam** untuk merepresentasikan satu siklus fermentasi penuh.

In [5]:
# Kondisi awal: [A₀, B₀, P₀]
init = [0.0, 0.0, 0.0]

# Titik waktu: 0 sampai 48 jam
t = np.linspace(0, 48, 500)

# Selesaikan sistem ODE secara numerik menggunakan odeint
y = odeint(network, init, t)

print("Simulasi selesai!")
print(f"Konsentrasi akhir pada t=48 jam:")
print(f"  [A] = {y[-1, 0]:.4f}")
print(f"  [B] = {y[-1, 1]:.4f}")
print(f"  [P] = {y[-1, 2]:.4f}")

Simulasi selesai!
Konsentrasi akhir pada t=48 jam:
  [A] = 0.3078
  [B] = 0.3898
  [P] = 28.4793


## Visualisasi Hasil Simulasi

Grafik berikut menampilkan dinamika konsentrasi ketiga metabolit internal (A, B, P) sepanjang siklus fermentasi 48 jam. Grafik bersifat **interaktif** dapat zoom dan pan untuk mengeksplorasi detail dinamika.

In [6]:
# Buat DataFrame untuk Altair
data = pd.DataFrame({
    'time': t,
    'A (Metabolit A)': y[:, 0],
    'B (Metabolit B)': y[:, 1],
    'P (Produk)':      y[:, 2]
})

# Ubah format ke long/melted untuk kompatibilitas Altair
data_melted = data.melt('time', var_name='Spesies', value_name='Konsentrasi')

# Buat grafik Altair dengan interaktivitas
chart = alt.Chart(data_melted).mark_line().encode(
    x=alt.X('time', title='Waktu (jam)'),
    y=alt.Y('Konsentrasi', title='Konsentrasi (mM)'),
    color=alt.Color('Spesies', title='Spesies',
                    scale=alt.Scale(range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
    tooltip=['time', 'Spesies', 'Konsentrasi']
).properties(
    title='Simulasi Dinamika Metabolik dengan Inhibisi Alosterik Non-Kompetitif',
    width=650,
    height=420
).interactive()  # Aktifkan zoom dan pan

chart.show()

alt.Chart(...)

## Penjelasan

### Komponen Utama Skrip

#### 1. Import Library
Skrip menggunakan beberapa library Python:
- **`scipy.integrate.odeint`**  menyelesaikan sistem ODE secara numerik menggunakan metode LSODA.
- **`numpy`**  operasi array dan komputasi numerik.
- **`pandas`**  manipulasi data tabular untuk kebutuhan visualisasi.
- **`altair`**  visualisasi data interaktif berbasis grammar of graphics.

---

#### 2. Definisi Sistem ODE

Reaksi dalam jaringan direpresentasikan sebagai:

$$X \xrightarrow{v_1} A \xrightarrow{v_2} B \xrightarrow{v_3} P$$
$$A \xrightarrow{v_4} \text{Byproduct}$$
$$P \dashrightarrow v_1 \text{ (inhibisi alosterik)}$$

Persamaan ODE untuk setiap metabolit internal:

$$\frac{d[A]}{dt} = v_1 - v_2 - v_4$$

$$\frac{d[B]}{dt} = v_2 - v_3$$

$$\frac{d[P]}{dt} = v_3$$

---

#### 3. Kondisi Awal

Konsentrasi awal semua metabolit internal ditetapkan nol karena pada awal fermentasi belum ada akumulasi intermediat:

$$[A]_0 = 0, \quad [B]_0 = 0, \quad [P]_0 = 0$$

Substrat X dijaga konstan pada nilai 10 (kondisi eksternal terkontrol).

---

#### 4. Penyelesaian ODE

Fungsi `odeint` digunakan untuk mengintegrasikan sistem ODE secara numerik selama rentang waktu 0–48 jam. Solver menghitung konsentrasi semua spesies pada setiap titik waktu menggunakan metode adaptif.

---

#### 5. Interpretasi Inhibisi Non-Kompetitif

Inhibitor (P) berikatan pada **situs alosterik** (bukan situs aktif), sehingga:
- Tidak mengubah afinitas enzim terhadap substrat (Kₘ tetap sama).
- Menurunkan kecepatan maksimal efektif (Vmax berkurang).
- Faktor inhibisi $(1 + [P]/K_i)$ mengalikan penyebut, menurunkan laju v₁ seiring meningkatnya [P].

$$v_1 = \frac{V_{1,max} \cdot [X]}{(K_{m1} + [X]) \cdot \left(1 + \frac{[P]}{K_i}\right)}$$

Ketika [P] = 0, persamaan ini menyederhanakan menjadi kinetika Michaelis-Menten standar:

$$v_1 = \frac{V_{1,max} \cdot [X]}{K_{m1} + [X]}$$

---

#### 6. Visualisasi Hasil

Grafik interaktif menampilkan:
- **[A]**  Metabolit A: naik cepat di awal, lalu mencapai *steady state* karena dikonsumsi oleh v₂ dan v₄.
- **[B]**  Metabolit B: naik mengikuti akumulasi A, lalu stabil.
- **[P]**  Produk P: terakumulasi secara bertahap, dan ketika cukup tinggi akan mulai menghambat v₁ melalui *feedback* alosterik.

Pola ini merepresentasikan **regulasi umpan balik negatif** yang umum dijumpai pada jalur biosintesis di sel mikroba.

---

### Referensi
- Lab Biotek UGM: https://github.com/lab-biotek-bio-ugm/S1_BISB211605_Biotechnology